# 09 - Model Versioning

## Objective
Save the winning model with full metadata so every prediction can be traced back to exactly which model made it.

Structure:
```
models/
  v1/
    attrition_pipeline.joblib
    metadata.json
```
---

In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from datetime import date
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

MODEL_PATH = "../models"
DATA_PATH = "../data/processed"

# Create version directory
version = "v1"
version_dir = f"{MODEL_PATH}/{version}"
os.makedirs(version_dir, exist_ok=True)

# Load pipeline
pipeline = joblib.load(f"{MODEL_PATH}/attrition_pipeline.joblib")
model = pipeline["model"]
scaler = pipeline["scaler"]
feature_names = pipeline["feature_names"]
model_name = pipeline["model_name"]

# Load data for evaluation
df = pd.read_csv(f"{DATA_PATH}/attrition_features.csv")
y = df["Attrition"]
X = df.drop(columns=["Attrition"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Get predictions
if scaler:
    X_test_input = scaler.transform(X_test)
else:
    X_test_input = X_test
y_pred = model.predict(X_test_input)
y_prob = model.predict_proba(X_test_input)[:, 1]

# Compute metrics
roc_auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Model: {model_name}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

Model: XGBoost
ROC-AUC: 0.7980
Precision: 0.5758
Recall: 0.4043
F1: 0.4750


---
## 1. Save Model Artifact
---

In [2]:
joblib.dump(pipeline, f"{version_dir}/attrition_pipeline.joblib")
print(f"Saved: {version_dir}/attrition_pipeline.joblib")

Saved: ../models/v1/attrition_pipeline.joblib


---
## 2. Create Metadata
---

In [3]:
metadata = {
    "model_name": "Attrition Prediction Model",
    "version": "v1.0",
    "algorithm": model_name,
    "training_date": str(date.today()),
    "dataset": "employee_attrition_processed.csv",
    "n_training_samples": len(X_train),
    "n_test_samples": len(X_test),
    "n_features": len(feature_names),
    "features": feature_names,
    "metrics": {
        "roc_auc": round(float(roc_auc), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1_score": round(float(f1), 4),
    },
    "confusion_matrix": {
        "true_negatives": int(cm[0][0]),
        "false_positives": int(cm[0][1]),
        "false_negatives": int(cm[1][0]),
        "true_positives": int(cm[1][1]),
    },
    "class_distribution": {
        "train_no": int((y_train == 0).sum()),
        "train_yes": int((y_train == 1).sum()),
        "test_no": int((y_test == 0).sum()),
        "test_yes": int((y_test == 1).sum()),
    },
    "preprocessing": {
        "scaling": "StandardScaler" if scaler else "None",
        "encoding": "OneHot + Binary mapping",
        "feature_engineering": ["income_per_year", "years_since_promotion_ratio", "satisfaction_score"],
    },
}

with open(f"{version_dir}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {version_dir}/metadata.json")
print()
print(json.dumps(metadata, indent=2))

Saved: ../models/v1/metadata.json

{
  "model_name": "Attrition Prediction Model",
  "version": "v1.0",
  "algorithm": "XGBoost",
  "training_date": "2026-09-01",
  "dataset": "employee_attrition_processed.csv",
  "n_training_samples": 1176,
  "n_test_samples": 294,
  "n_features": 44,
  "features": [
    "Age",
    "DistanceFromHome",
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "MonthlyIncome",
    "NumCompaniesWorked",
    "OverTime",
    "PercentSalaryHike",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TotalWorkingYears",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager",
    "income_per_year",
    "years_since_promotion_ratio",
    "satisfaction_score",
    "BusinessTravel_Travel_Frequently",
    "BusinessTravel_Travel_Rarely",
    "Department_Research & Development",

---
## 3. Verify Saved Artifacts
---

In [4]:
print(f"=== Saved artifacts in {version_dir}/ ===")
for f in sorted(os.listdir(version_dir)):
    size = os.path.getsize(f"{version_dir}/{f}")
    print(f"  {f}: {size:,} bytes")

# Reload and verify
loaded = joblib.load(f"{version_dir}/attrition_pipeline.joblib")
print(f"\nReloaded model type: {type(loaded['model']).__name__}")
print(f"Reloaded features: {len(loaded['feature_names'])}")

# Verify metadata
with open(f"{version_dir}/metadata.json", "r") as f:
    meta = json.load(f)
print(f"Metadata version: {meta['version']}")
print(f"Algorithm: {meta['algorithm']}")
print(f"ROC-AUC: {meta['metrics']['roc_auc']}")
print(f"F1: {meta['metrics']['f1_score']}")

=== Saved artifacts in ../models/v1/ ===
  attrition_pipeline.joblib: 504,000 bytes
  metadata.json: 2,155 bytes

Reloaded model type: XGBClassifier
Reloaded features: 44
Metadata version: v1.0
Algorithm: XGBoost
ROC-AUC: 0.798
F1: 0.475


---
## Versioning Summary

| Artifact | Path |
| --- | --- |
| Model pipeline | `models/v1/attrition_pipeline.joblib` |
| Metadata | `models/v1/metadata.json` |
| Model type | {algorithm} |
| ROC-AUC | {roc_auc} |
| F1 | {f1} |

Every future prediction can be traced to this exact model version.

**Day 2 complete.** Awaiting instruction before starting Day 3 (Workforce Intelligence).


## MLflow Integration

Track this model version in MLflow for automatic parameter/metric/artifact logging.

In [5]:
import mlflow

import mlflow.sklearn

import os



# Use SQLite database backend (MLflow 3.x requirement)

db_path = os.path.abspath("../mlflow.db")

mlflow.set_tracking_uri(f"sqlite:///{db_path}")

mlflow.set_experiment("enterprise_hr_ai_attrition")



with mlflow.start_run(run_name="v1_xgboost"):

    mlflow.log_param("algorithm", "XGBoost")

    mlflow.log_param("n_features", len(feature_names))

    mlflow.log_param("version", "v1.0")

    

    mlflow.log_metric("roc_auc", meta["metrics"]["roc_auc"])

    mlflow.log_metric("f1_score", meta["metrics"]["f1_score"])

    mlflow.log_metric("precision", meta["metrics"]["precision"])

    mlflow.log_metric("recall", meta["metrics"]["recall"])

    

    mlflow.log_artifact("../models/attrition_pipeline.joblib")

    mlflow.log_artifact("../models/v1/metadata.json")

    

    run_id = mlflow.active_run().info.run_id

    print(f"MLflow run logged: {run_id}")

    print(f"Tracking URI: sqlite:///{db_path}")

    print(f"Metrics: ROC-AUC={meta["metrics"]["roc_auc"]}, F1={meta["metrics"]["f1_score"]}")

2026/09/01 13:15:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/01 13:15:12 INFO mlflow.store.db.utils: Updating database tables


2026/09/01 13:15:14 INFO mlflow.tracking.fluent: Experiment with name 'enterprise_hr_ai_attrition' does not exist. Creating a new experiment.


MLflow run logged: a81dfcabd5594baba5703dcf9f8c1628
Tracking URI: sqlite:///E:\Agentic Project2\enterprise_hr_ai\mlflow.db
Metrics: ROC-AUC=0.798, F1=0.475
